# **Training Tiny LLaMA with QLoRA**

This notebook demonstrates a two-step approach to fine-tuning generative Large Language Models (LLMs) using memory-efficient techniques - Supervised Fine-Tuning (SFT) and Preference Tuning (DPO). Specifically, it uses QLoRA (Quantized Low-Rank Adaptation) to perform both instruction tuning and alignment on a small model, TinyLlama-1.1B.

### **Install the required modules**

**Massive Speed Boost:** Every time you run a separate !pip install command, Python has to wake up, check your entire system for existing packages, resolve dependencies, close out, and then repeat the exact same process for the next line. Combining them allows pip to scan your system and install everything in one single pass, saving you a lot of time in Colab.

**Dependency Resolution:** Deep learning packages are notoriously picky about working with each other. If you install them one by one, a later package might silently uninstall or downgrade an earlier package to satisfy its own requirements. Putting them in one command forces pip to look at all the versions simultaneously and find a mathematical compromise that keeps everything stable.

**The Backslash (\) Trick:** Using the backslash allows you to break a single long command into a neat vertical list. This keeps it highly readable while technically remaining a single command. (Just make sure there are no spaces after the \ or it will error out!)

In [ ]:
!pip install -qU \
accelerate==0.31.0 \
peft==0.11.1 \
bitsandbytes==0.43.1 \
transformers==4.41.2 \
trl==0.9.4 \
sentencepiece==0.2.0 \
triton==3.1.0 \
torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

  torchvision==0.27.1

**Restart the session after all the modules are installed.**

###**Import the required Libraries**

In [ ]:
# Standard Deep Learning and Hardware Framework
import torch
import transformers
import accelerate
import peft
import trl
import torchvision

# Hugging Face Datasets
from datasets import load_dataset

# Hugging Face Transformers & Pipelines
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, TrainingArguments

# PEFT (Parameter-Efficient Fine-Tuning) / LoRA Tools
from peft import AutoPeftModelForCausalLM, LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model

# TRL (Transformer Reinforcement Learning) Trainers & Configs
from trl import SFTTrainer, DPOConfig, DPOTrainer

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("torchvision:", torchvision.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.5.1+cu124
transformers: 4.41.2
peft: 0.11.1
trl: 0.9.4
torchvision: 0.20.1+cu124
accelerate: 0.31.0


##**Part 1: Supervised Fine-Tuning (SFT)**

###**Data Processing**

####**Load the Dataset**

We won't use the full dataset. We are extracting it just to show that the load_dataset function returns a DatasetDictionary when used as is. But when the split is specified it returns the specific Dataset within the Dataset Dictionary, which in turn may be a dictionary item of list, as is in this case.

In [ ]:
dataset_full = (
  load_dataset("HuggingFaceH4/ultrachat_200k")
)

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-f7dfac4afe5(…):   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

data/test_gen-00000-of-00001-3d4cd830914(…):   0%|          | 0.00/80.4M [00:00<?, ?B/s]

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

In [ ]:
dataset_full

DatasetDict({
    train_sft: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 207865
    })
    test_sft: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 23110
    })
    train_gen: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 256032
    })
    test_gen: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 28304
    })
})

Now we are specifying the split. This is the dataset that we will use for training our model.

In [ ]:
dataset = (
  load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")
    .shuffle(seed=42)
    .select(range(3_000))
)

In [ ]:
dataset

Dataset({
    features: ['prompt', 'prompt_id', 'messages'],
    num_rows: 3000
})

Note that it now returns a Dataset Item, not a Dataset Dictionary. It is memory-efficient and can handle datasets larger than your RAM.


*   features: ['prompt', 'prompt_id', 'messages']: These are the column names (or fields) in your dataset.
*   num_rows: 3000: There are exactly 3,000 individual rows (or samples) of data in this specific dataset.

Access a specific row (e.g., the 2577th row), Note that indexing begins at 0.

In [ ]:
dataset[2576]

{'prompt': 'Given the text: Knock, knock. Who’s there? Hike.\nCan you continue the joke based on the given text material "Knock, knock. Who’s there? Hike"?',
 'prompt_id': '105efc069a9a11cd927a80fcefe3c43dc2a9c5a4c237131a81cd37d47e2e3454',
 'messages': [{'content': 'Given the text: Knock, knock. Who’s there? Hike.\nCan you continue the joke based on the given text material "Knock, knock. Who’s there? Hike"?',
   'role': 'user'},
  {'content': "Sure! Knock, knock. Who's there? Hike. Hike who? Hike up your pants, it's cold outside!",
   'role': 'assistant'},
  {'content': 'Can you tell me another knock-knock joke based on the same text material "Knock, knock. Who\'s there? Hike"?',
   'role': 'user'},
  {'content': "Of course! Knock, knock. Who's there? Hike. Hike who? Hike your way over here and let's go for a walk!",
   'role': 'assistant'}]}

Check the exact data types of the features

In [ ]:
dataset.features

{'prompt': Value('string'),
 'prompt_id': Value('string'),
 'messages': List({'content': Value('string'), 'role': Value('string')})}

Access an entire column

In [ ]:
dataset["prompt_id"]

Column(['a59f5664bebecec40565355945c2b124b40b891096d9dfc807812fe33c7b8e47', '8d754cc700a2a07656c27e6fa00916efbfa328fa98a151c63dceeee8b8d3c68a', 'bea86e56e110ef3dc4992d794488a72adb943b54f35b0ba67a51a8e0ff93107b', 'a336983f0be7a0cc0e2c567160f5e8e23d191b203c38079028220141064c4dd0', 'a3157f4f7834ac6841972db79891c6a67bb9495b9b2c5d7a1d421c44ea882abb'])

In [ ]:
dataset["messages"]

Column([[{'content': "Write a compelling mystery story set in a vineyard, where a seasoned detective investigates a murder with twists and turns that will keep the reader engaged until the very end. Add complex characters, multiple suspects, and red herrings to create suspense and challenge the detective's deductive reasoning. Use vivid descriptive language to paint a picture of the vineyard setting, its wine-making process, and the people who live and work there. Make sure to reveal clues and motives gradually, and create a satisfying resolution that ties up all loose ends.", 'role': 'user'}, {'content': "Detective Jameson had been called to the vineyard on the outskirts of town to investigate a murder. The sun was setting, casting long shadows over the grape vines, and the air was heavy with the sweet scent of fermented grapes. The body was lying in the middle of the vineyard, surrounded by broken grape vines and a scattering of grapes. The victim's throat had been slashed, and there

Access the first message of the 2577th row

In [ ]:
dataset[2576]["messages"][0]

{'content': 'Given the text: Knock, knock. Who’s there? Hike.\nCan you continue the joke based on the given text material "Knock, knock. Who’s there? Hike"?',
 'role': 'user'}

Convert the entire dataset to a Pandas DataFrame if you prefer working with tables

In [ ]:
df = dataset.to_pandas()
df.head()

,prompt,prompt_id,messages
0,Write a compelling mystery story set in a vine...,a59f5664bebecec40565355945c2b124b40b891096d9df...,[{'content': 'Write a compelling mystery story...
1,Could you provide a recipe for a homemade vega...,8d754cc700a2a07656c27e6fa00916efbfa328fa98a151...,[{'content': 'Could you provide a recipe for a...
2,Does the University of Pennsylvania offer any ...,bea86e56e110ef3dc4992d794488a72adb943b54f35b0b...,[{'content': 'Does the University of Pennsylva...
3,Can you summarize the activities and attractio...,a336983f0be7a0cc0e2c567160f5e8e23d191b203c3807...,[{'content': 'Can you summarize the activities...
4,Write a futuristic science fiction story about...,a3157f4f7834ac6841972db79891c6a67bb9495b9b2c5d...,[{'content': 'Write a futuristic science ficti...


So now you fully understand the dataset!

####**Load a tokenizer to use its chat template**

It pulls a tokenizer from a chat-tuned version of TinyLlama to use its built-in chat template.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

The Jinja2 template is stored in the chat_template attribute of the loaded tokenizer.

In [ ]:
print(tokenizer.chat_template)

{% for message in messages %}
{% if message['role'] == 'user' %}
{{ '<|user|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'system' %}
{{ '<|system|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'assistant' %}
{{ '<|assistant|>
'  + message['content'] + eos_token }}
{% endif %}
{% if loop.last and add_generation_prompt %}
{{ '<|assistant|>' }}
{% endif %}
{% endfor %}


Test with Dummy Messages

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there!"},
]

# Set add_generation_prompt=True to see what gets appended for the model's next reply
formatted_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(formatted_text)

<|system|>
You are a helpful assistant.</s>
<|user|>
Hello!</s>
<|assistant|>
Hi there!</s>
<|assistant|>



####**The Mapping Function**

In [ ]:
def format_prompt(row):
  """Format the prompt to using the <|user|> template TinyLLama is using"""

  # Format answers
  chat = row["messages"]
  prompt = tokenizer.apply_chat_template(chat, tokenize=False)

  return {"text": prompt}

The Hugging Face apply_chat_template() fuction reads the chat list, looks at the "role", and wraps the "content" in the exact tags TinyLlama needs. Because tokenize=False is specified, it returns a plain, readable string rather than converting the words into math numbers (tokens) just yet.

After this line runs, the raw dictionary list is transformed into a single unified text string that looks like this:



```
<|user|>
Given the text: Knock, knock. Who’s there? Hike. Can you continue the joke...?</s>
<|assistant|>
Sure! Knock, knock. Who's there? Hike. Hike who? Hike up your pants...</s>
```

The Hugging Face .map() function expects data modifications to be returned as a dictionary.



```
return {"text": prompt}
```


This line takes the newly formatted string, packages it into a new column called "text", and adds it back to the dataset so the trainer can read it sequentially during the fine-tuning phase.


In [ ]:
dataset = dataset.map(format_prompt)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

The map() function in Hugging Face's datasets library applies a transformation function (like format_prompt) to every row in the dataset, returning a new dataset with the updated or added fields.


*   Iterates Over Rows: passes each row as a Python dictionary (row) to format_prompt
*   Executes Logic: format_prompt reads the existing fields (e.g., row["messages"]), applies apply_chat_template, and returns a dictionary {"text": prompt}
*   Appends/Updates Columns: The returned keys become columns in the dataset. If a returned key already exists, it is overwritten; if it is new (like "text"), it is added as a new column alongside the original ones.

Let's see what the dataset now contains.

In [ ]:
dataset

Dataset({
    features: ['prompt', 'prompt_id', 'messages', 'text'],
    num_rows: 3000
})

Notice that one more column "text" got added to dataset.

In [ ]:
dataset["text"]

Column(["<|user|>\nWrite a compelling mystery story set in a vineyard, where a seasoned detective investigates a murder with twists and turns that will keep the reader engaged until the very end. Add complex characters, multiple suspects, and red herrings to create suspense and challenge the detective's deductive reasoning. Use vivid descriptive language to paint a picture of the vineyard setting, its wine-making process, and the people who live and work there. Make sure to reveal clues and motives gradually, and create a satisfying resolution that ties up all loose ends.</s>\n<|assistant|>\nDetective Jameson had been called to the vineyard on the outskirts of town to investigate a murder. The sun was setting, casting long shadows over the grape vines, and the air was heavy with the sweet scent of fermented grapes. The body was lying in the middle of the vineyard, surrounded by broken grape vines and a scattering of grapes. The victim's throat had been slashed, and there were bruises a

Let's grab the 2577th row of the "text" column

In [ ]:
print(dataset["text"][2576])

<|user|>
Given the text: Knock, knock. Who’s there? Hike.
Can you continue the joke based on the given text material "Knock, knock. Who’s there? Hike"?</s>
<|assistant|>
Sure! Knock, knock. Who's there? Hike. Hike who? Hike up your pants, it's cold outside!</s>
<|user|>
Can you tell me another knock-knock joke based on the same text material "Knock, knock. Who's there? Hike"?</s>
<|assistant|>
Of course! Knock, knock. Who's there? Hike. Hike who? Hike your way over here and let's go for a walk!</s>



So the chat texts are now demarcated with <|user|> and <|assistant|>.

So we have converted our training dataset to the required format.

Let's focus on quantization now.

###**Model Quantization**

Load the base 1.1-billion-parameter TinyLlama model into consumer-grade GPU memory by crushing its weights from 16-bit floating-point numbers down to 4-bit NormalFloat. This drastically reduces VRAM consumption, allowing the model to be trained on a standard GPU (like a free Google Colab T4 instance) without sacrificing much accuracy.

In [ ]:
model_name="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,  # Use 4-bit precision model loading
  bnb_4bit_quant_type="nf4",  # Quantization type
  bnb_4bit_compute_dtype="float16",  # Compute dtype
  bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

In [ ]:
# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
  model_name,
  device_map="auto",

  # Leave this out for regular SFT
  quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [ ]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

While you load the model to handle the heavy mathematical weights, you load the tokenizer to act as the translator that converts raw human words into numerical numbers (tokens) that the model can actually compute.

When you train an AI model, you feed it data in batches (e.g., processing 2 or 4 sentences at the exact same time) to maximize GPU speed. However, sentences are naturally different lengths. Neural networks require every input in a batch to be the exact same matrix size. So Padding is used with the token <PAD> , at the very beginning (left side) of the text sequence.

LLM models like TinyLlama are causal (autoregressive), meaning they read text from left to right and predict the very next token based entirely on the tokens that came right before it. If you used standard right-padding, your prompt would end with a massive wall of meaningless <PAD> tokens. The model looks at the final token (<PAD>) and tries to predict what comes after a padding placeholder, resulting in broken logic or endless gibberish loops.

By padding on the left, the dummy tokens are pushed to the front, ensuring the active text ends exactly where the model needs to start generating. The model reads the clean ending and immediately starts generating a relevant response!

###**LoRA Configuration**

This block of code sets up LoRA (Low-Rank Adaptation), which is the secret sauce that allows you to fine-tune a massive 1.1-billion-parameter model on a weak GPU.

Instead of updating all the heavy weights of the base model (which takes massive amounts of memory), LoRA freezes the original model and injects tiny, lightweight "adapter" layers. Only these tiny adapters get trained.

In [ ]:
# Prepare LoRA Configuration
peft_config = LoraConfig(
  lora_alpha=32,  # LoRA Scaling
  lora_dropout=0.1,  # Dropout for LoRA Layers
  r=64,  # Rank
  bias="none",
  task_type="CAUSAL_LM",
  target_modules=  # Layers to target
  ["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj",
  "down_proj"]
  )

In [ ]:
# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

###**Training Configuration**

In [ ]:
output_dir = "./results"

# Training arguments
training_arguments = TrainingArguments(
  output_dir=output_dir,
  per_device_train_batch_size=2,
  gradient_accumulation_steps=4,
  optim="paged_adamw_32bit",
  learning_rate=2e-4,
  #report_to="none", #Turn off wandb reporting
  lr_scheduler_type="cosine",
  num_train_epochs=1,
  logging_steps=10,
  fp16=True,
  gradient_checkpointing=True
  )

###**Training**

This block of code initializes the SFTTrainer (Supervised Fine-Tuning Trainer) from the trl library.

Think of this as the orchestrator of your entire project. Up until this point, you have created separate puzzle pieces: the dataset, the 4-bit model, the LoRA configuration adapters, and the training math settings. The SFTTrainer acts as the manager that takes all of these components, hooks them together, and runs the actual machine learning loop.

In [ ]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
  model=model,
  train_dataset=dataset,
  dataset_text_field="text",
  tokenizer=tokenizer,
  args=training_arguments,
  max_seq_length=512,
  # Leave this out for regular SFT
  peft_config=peft_config,
  )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1965: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:307: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:397: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
# Train model
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gaurav-dey-2024 (gaurav-dey-2024-iiit-dharwad) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.671300
20,1.475900
30,1.451200
40,1.488400
50,1.477800
60,1.390700
70,1.494900
80,1.450200
90,1.427300
100,1.404300


TrainOutput(global_step=375, training_loss=1.4169210255940756, metrics={'train_runtime': 1783.2285, 'train_samples_per_second': 1.682, 'train_steps_per_second': 0.21, 'total_flos': 9994755938844672.0, 'train_loss': 1.4169210255940756, 'epoch': 1.0})

Because you used LoRA, the core base model weights were frozen and completely untouched. The only things that changed were the tiny injected adapter layers.

When you run trainer.model.save_pretrained(), the script is smart: it skips the 4.4 GB base model entirely and only saves your lightweight adapter weights. The resulting folder (TinyLlama-1.1B-qlora) will be incredibly small—usually only around 50 to 100 Megabytes.

If you open your file explorer in Colab and look inside the new "TinyLlama-1.1B-qlora" directory, you will find a few small files

In [ ]:
# Save QLoRA weights
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


###**Merge Adapter**

This block of code is where you prepare your freshly trained model for actual deployment or inference.

Up until this line, your model has been split into two pieces: a frozen 4-bit base model and a separate set of lightweight LoRA adapters. This code loads those adapters back up and permanently fuses (merges) them directly into the base model structure.

Once the changes are fused, it completely deletes (unloads) the separate LoRA adapter structures from your GPU memory.

In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload()

**Why Is Merging Essential?**

Running a split model requires extra processing overhead because data has to route through multiple matrices sequentially. Fusing them together means your model runs at its absolute maximum native speed during text generation.

A merged model is no longer a "special PEFT model." It is now a standard standalone causal language model. You can save it to your hard drive, deploy it using production frameworks (like vLLM, Ollama, or Hugging Face pipelines), and it will behave exactly like any standard out-of-the-box model, just with your custom instruction-tuned intelligence built directly into its core weights!


**Using the Model**

In [ ]:
# Use our predefined prompt template
prompt = """<|user|>
Tell me something about Guwahati.</s>
<|assistant|>
"""

# Run our instruction-tuned model
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

<|user|>
Tell me something about Guwahati.</s>
<|assistant|>
Guwahati is a city in the Indian state of Assam. It is located on the banks of the Brahmaputra River and is the largest city in the state. Guwahati is known for its scenic beauty, historical monuments, and cultural heritage. The city is also known for its shopping and dining options, and is a popular tourist destination.


###**Save and Download the Merged Model to your Local Hard drive**

To save your merged model from Google Colab directly to your local hard drive, you need to follow a two-step process. First, you save the model files into Colab's temporary cloud storage, and then you download them to your physical computer.

Because a 1.1B parameter model results in a folder of files totaling about 4.4 Gigabytes, downloading via a web browser can sometimes be unstable.

Here are the two best methods to get the model onto your local machine

####**Method 1: Save and Download Directly**

**Step 1. Save the model and the tokenizer files to a local directory inside Colab Workspace**

This command takes the model from the GPU's volatile memory and writes the raw files (the weights and configurations) into a folder inside Colab.

In [ ]:
merged_model.save_pretrained("./my_local_tinyllama")
tokenizer.save_pretrained("./my_local_tinyllama")

**Step 2: Zip the folder and download it**

In [ ]:
import shutil
from google.colab import files

# 1. Compress the folder into a single .zip file
shutil.make_archive("my_local_tinyllama", 'zip', "./my_local_tinyllama")

# 2. Trigger your web browser to download the zip file to your computer
files.download("my_local_tinyllama.zip")

####**Method 2: Save via Google Drive (Most Reliable)**

**Step 1: Mount your Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Step 2: Save the model straight to your Drive**

In [ ]:
# Create a dedicated path inside your Google Drive
target_drive_path = "/content/drive/MyDrive/my_local_tinyllama"

# Save directly to your Google Drive
merged_model.save_pretrained(target_drive_path)
tokenizer.save_pretrained(target_drive_path)
print("Model safely saved to your Google Drive!")

Model safely saved to your Google Drive!


**Step 3: Grab it on your computer**

Open a new tab on your computer and go to Google Drive.

Find the folder named my_local_tinyllama.

Right-click the folder and select Download. Google Drive will automatically zip the folder on its servers and download a clean copy straight to your local hard drive.

###**How to use it on your Local Computer Later**

Once the folder is unzipped on your local machine (let's say you extracted it to D:/models/my_local_tinyllama), you can load your custom fine-tuned model completely offline in your local Python scripts without downloading anything from the internet.



```
from transformers import AutoModelForCausalLM, AutoTokenizer

local_path = "D:/models/my_local_tinyllama"  # Path on your actual hard drive

tokenizer = AutoTokenizer.from_pretrained(local_path)
model = AutoModelForCausalLM.from_pretrained(local_path)
```

